In [2]:
%pip install --upgrade --quiet google-cloud-aiplatform datasets scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys

import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [2]:
import traceback
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from optparse import Option
from typing import Optional, List, Callable, Any, Dict

import vertexai
import os
from datasets import load_dataset
from google.api_core.exceptions import ResourceExhausted
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm import tqdm
import pandas as pd
from google.cloud import aiplatform
import multiprocess as mp

from sklearn.model_selection import train_test_split
from vertexai.generative_models import GenerationConfig, HarmCategory, HarmBlockThreshold, GenerativeModel
import backoff

In [3]:
#######################
# Initialize Vertex AI
#######################
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/home/rajan/CREDENTIALS/rtc-ai-20240203-c915144c2c55.json"
PROJECT_ID = "rtc-ai-20240203"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
vertexai.init(project=PROJECT_ID, location=LOCATION)



In [4]:
#############################
# Load and split the dataset
#############################
bbc_datasets = load_dataset("SetFit/bbc-news")

train = pd.DataFrame(bbc_datasets["train"])
test = pd.DataFrame(bbc_datasets["test"])

# Remove the row containing the specified text
text_to_remove = "jackson film  absolute disaster  a pr expert has told the michael jackson child abuse trial that the tv"
test = test[~test['text'].str.contains(text_to_remove, na=False)]

print(f"Training Dataset head: \n {train.head()}")
print(f"Testing Dataset head: \n {test.head()}")

print(f"Training Dataset shape: {train.shape}")
print(f"Testing Dataset shape: {test.shape}")

val, test = train_test_split(
    test, test_size=0.75, shuffle=True, random_state=2, stratify=test["label_text"]
)
print(f"Validation Dataset shape: {val.shape}")
print(f"Testing Dataset shape: {test.shape}")

print(f"Validation Dataset Labels' count: {val.label_text.value_counts()}")
print(f"Testing Dataset Lables' count: {test.label_text.value_counts()}")




Training Dataset head: 
                                                 text  label     label_text
0  wales want rugby league training wales could f...      2          sport
1  china aviation seeks rescue deal scandal-hit j...      1       business
2  rock band u2 break ticket record u2 have smash...      3  entertainment
3  markets signal brazilian recovery the brazilia...      1       business
4  tough rules for ringtone sellers firms that fl...      0           tech
Testing Dataset head: 
                                                 text  label     label_text
0  carry on star patsy rowlands dies actress pats...      3  entertainment
1  sydney to host north v south game sydney will ...      2          sport
2  uk coal plunges into deeper loss shares in uk ...      1       business
3  blair joins school sailing trip the prime mini...      4       politics
4  bath faced with tindall ultimatum mike tindall...      2          sport
Training Dataset shape: (1225, 3)
Testing Dataset s

In [84]:
###################
# Experiment Setup
###################
class VertexAIExperimentManager:

    def __init__(self, project: str, location: str):
        self.project = project
        self.location = location
        self.current_experiment = None

    def init_experiment(self, experiment_name: str, experiment_description: Optional[str] = None):
        self.current_experiment = experiment_name
        aiplatform.init(
            experiment=experiment_name,
            experiment_description=experiment_description,
            experiment_tensorboard=False,
            project=self.project,
            location=self.location)

    def create_experiment(self, experiment_name: str, experiment_description: Optional[str] = None):
        self.init_experiment(experiment_name, experiment_description)

    def log_run(self, run_name: str, params: Dict[str, Any], metrics: Dict[str, Any]) -> None:
        if not self.current_experiment:
            raise ValueError("Experiment not initialized")

        aiplatform.start_run(run=run_name)
        aiplatform.log_params(params)
        aiplatform.log_metrics(metrics)
        aiplatform.end_run()

    def get_experiments_data_frame(self) -> Optional[pd.DataFrame]:
        """Retrieve DataGrame of experiment data from Vertex AI experiements"""
        if not self.current_experiment:
            raise ValueError("Experiment not initialized. Call init_experiment first.")        

        return aiplatform.get_experiment_df()



In [86]:
EXPERIMENT_NAME = "student-experiment-1"
experiment_manager = VertexAIExperimentManager(project=PROJECT_ID, location=LOCATION)
experiment_manager.create_experiment(
    experiment_name=EXPERIMENT_NAME,
    experiment_description="Gemini Fine Tuning demo experiment"
)




In [7]:

system_prompt_zero_shot = """TASK:
Classify the text into ONLY one of the following classes [business, entertainment, politics, sport, tech].

CLASSES:
- business
- entertainment
- politics
- sport
- tech

INSTRUCTIONS
- Respond with ONLY one class.
- You MUST use the exact word from the list above.
- DO NOT create or use any other classes.
- CAREFULLY analyze the text before choosing the best-fitting category from [business, entertainment, politics, sport, tech].

"""

system_prompt_few_shot = f"""TASK:
Classify the text into ONLY one of the following classes [business, entertainment, politics, sport, tech].

CLASSES:
- business
- entertainment
- politics
- sport
- tech

INSTRUCTIONS:
- Respond with ONLY one class.
- You MUST use the exact word from the list above.
- DO NOT create or use any other classes.
- CAREFULLY analyze the text before choosing the best-fitting category from [business, entertainment, politics, sport, tech].

EXAMPLES:
- EXAMPLE 1:
    <user>
    {train.loc[train["label_text"] == "business", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "business", "label_text"].iloc[10]}

- EXAMPLE 2:
    <user>
    {train.loc[train["label_text"] == "entertainment", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "entertainment", "label_text"].iloc[10]}

- EXAMPLE 3:
    <user>
    {train.loc[train["label_text"] == "politics", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "politics", "label_text"].iloc[10]}

- EXAMPLE 4:
    <user>
    {train.loc[train["label_text"] == "sport", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "sport", "label_text"].iloc[10]}

- EXAMPLE 5:
    <user>
    {train.loc[train["label_text"] == "tech", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "tech", "label_text"].iloc[10]}

"""



In [8]:
######################
# Model Configuration
######################
generation_config = GenerationConfig(max_output_tokens=10, temperature=0)
safety_settings = {
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
}

gem_pro_1_model_zero = GenerativeModel(
    "gemini-1.0-pro-002",  # e.g. gemini-1.5-pro-001, gemini-1.5-flash-001
    system_instruction=[system_prompt_zero_shot],
    generation_config=generation_config,
    safety_settings=safety_settings,
)



In [9]:
####################################################
# Utility Function: Batch Prediction (Parallelism)
####################################################

def backoff_hdlr(details) -> None:
    print(f"Backing off {details['wait']} seconds after {details['tries']} tries.")

def log_error(msg: str, *args: Any):
    mp.get_logger().error(msg, *args)
    raise Exception(msg)

def handle_exception_threading(f: Callable) -> Callable:
    def applicator(*args, **kwargs):
        try:
            return f(*args, **kwargs)
        except:
            log_error(traceback.format_exc())

    return applicator

@handle_exception_threading
@backoff.on_exception(
    backoff.expo, ResourceExhausted, max_tries=30, on_backoff=backoff_hdlr
)
def _predict_message(message: str, model: GenerativeModel) -> Optional[str]:
    response = model.generate_content([message], stream=False)

    # TODO: Take care of LLM safety aspects
    response_dict = response.to_dict()

    # Check if "prompt_feedback" exists and has the key "block_reason"
    try:
        prompt_feedback = response_dict.get("prompt_feedback", {})
        block_reason = prompt_feedback.get("block_reason")

        if block_reason == "PROHIBITED_CONTENT":
            # Handle the case for prohibited content
            print("Block Reason: ", response.prompt_feedback.block_reason_message)
            print("Message: ", message)
            return response.prompt_feedback.block_reason_message
        else:
            # Handle other cases or continue processing
            return response.text
    except Exception as e:
        # Log the exception or handle errors accordingly
        print(f"An error occurred: {e}")

def batch_predict(
        messages: List[str],
        model: GenerativeModel,
        max_workers: int = 4
) -> List[Optional[str]]:

    predictions = list()

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        partial_func = partial(_predict_message, model=model)
        for prediction in tqdm(pool.map(partial_func, messages), total=len(messages)):
            predictions.append(prediction)

    return predictions


def predictions_postprocessing(text: str) -> str:
    return text.strip().lower()

def evaluate_predictions(
    df: pd.DataFrame,
    target_column: str = "label_text",
    prediction_column: str = "prediction_labels",
    postprocessing: bool = True,
) -> Dict[str, float]:

    if postprocessing:
        df[prediction_column] = df[prediction_column].apply(predictions_postprocessing)

    y_true = df[target_column]
    y_pred = df[prediction_column]

    metrics_report = classification_report(y_true, y_pred, output_dict=True)
    overall_macro_f1_score = f1_score(y_true, y_pred, average="macro")
    overall_micro_f1_score = f1_score(y_true, y_pred, average="micro")
    weighted_precision = precision_score(y_true, y_pred, average="weighted")
    weighted_recall = recall_score(y_true, y_pred, average="weighted")

    metrics = {
        "accuracy": metrics_report["accuracy"],
        "weighted precision": weighted_precision,
        "weighted recall": weighted_recall,
        "macro f1 score": overall_macro_f1_score,
        "micro f1 score": overall_micro_f1_score
    }

    categories = ["business", "entertainment", "politics", "sport", "tech"]
    for category in categories:
        if category in metrics_report:
            metrics[f"{category}_f1_score"] = metrics_report[category]["f1-score"]

    return metrics



#######################################################################


In [10]:
messages_to_predict = test["text"].to_list()

In [11]:

predictions_zero_shot = batch_predict(
    messages=messages_to_predict, model=gem_pro_1_model_zero, max_workers=4
)

print(predictions_zero_shot)



 42%|███████████████████████████████████████████████████████████████████████▊                                                                                                    | 313/750 [00:54<01:05,  6.67it/s]

Backing off 0.2936510978010075 seconds after 1 tries.
Backing off 0.8434634079896906 seconds after 1 tries.
Backing off 0.05385370944387513 seconds after 1 tries.


 42%|████████████████████████████████████████████████████████████████████████▏                                                                                                   | 315/750 [00:55<01:21,  5.33it/s]

Backing off 0.49232298280242426 seconds after 2 tries.
Backing off 1.3817287878918123 seconds after 2 tries.
Backing off 0.17346089295536926 seconds after 1 tries.
Backing off 1.5627474411926183 seconds after 2 tries.
Backing off 1.5983650771438314 seconds after 2 tries.
Backing off 1.9502475063907583 seconds after 3 tries.
Backing off 1.0729406807174913 seconds after 3 tries.
Backing off 1.5610276764394455 seconds after 3 tries.
Backing off 2.3221507889973094 seconds after 3 tries.
Backing off 7.436478643050766 seconds after 4 tries.
Backing off 0.019628478210846012 seconds after 4 tries.
Backing off 12.376076889750129 seconds after 5 tries.
Backing off 2.3450896653564683 seconds after 4 tries.
Backing off 1.473229806914497 seconds after 4 tries.
Backing off 10.76394043880474 seconds after 5 tries.
Backing off 4.731774827859644 seconds after 5 tries.
Backing off 12.103829501582974 seconds after 5 tries.
Backing off 19.86193830828326 seconds after 6 tries.
Backing off 18.25586341452197

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 750/750 [02:52<00:00,  4.35it/s]

['sport', 'politics', 'sport', 'tech', 'politics', 'business', 'business', 'entertainment', 'politics', 'sport', 'politics', 'politics', 'tech', 'business', 'sport', 'politics', 'politics', 'entertainment \n', 'tech', 'politics', 'politics', 'entertainment', 'sport', 'sport', 'entertainment \n', 'tech', 'politics', 'entertainment', 'politics', 'entertainment', 'sport', 'politics', 'entertainment', 'sport', 'sport', 'entertainment', 'tech', 'sport', 'sport', 'entertainment', 'sport', 'entertainment', 'politics', 'sport', 'entertainment', 'sport', 'entertainment', 'business', 'business', 'entertainment', 'business', 'politics', 'sport', 'politics', 'entertainment', 'sport', 'tech', 'business', 'politics', 'business', 'sport', 'sport', 'sport', 'politics', 'business', 'business', 'tech', 'sport', 'politics', 'sport', 'sport', 'tech', 'sport', 'politics', 'sport', 'politics', 'tech', 'business', 'business', 'politics', 'politics', 'business', 'tech', 'entertainment', 'sport', 'entertainmen

In [12]:
# Create an Evaluation dataframe to store the predictions for all the experiments.
df_evals = test.copy()
df_evals["gem1.0-zero-shot_predictions"] = predictions_zero_shot

# Compute Evaluation Metrics for zero-shot prompt
metrics_zero_shot = evaluate_predictions(
    df_evals.copy(),
    target_column = "label_text",
    prediction_column = "gem1.0-zero-shot_predictions",
    postprocessing = True
)
print(metrics_zero_shot)



{'accuracy': 0.9146666666666666, 'weighted precision': 0.9208293722158309, 'weighted recall': 0.9146666666666666, 'macro f1 score': 0.9122658578409164, 'micro f1 score': 0.9146666666666666, 'business_f1_score': 0.896551724137931, 'entertainment_f1_score': 0.9323308270676691, 'politics_f1_score': 0.8758620689655172, 'sport_f1_score': 0.9668508287292817, 'tech_f1_score': 0.8897338403041825}


In [ ]:
# Log Experiment with zero-shot prompt with Gemini 1.0 pro
params = {
    "model": "gemini-1.0-pro-002",
    "adaptation_type": "in-context zero-shot",
    "temperature": 0,
    "max_output_tokens": 10,
}


experiment_manager.log_run(
    run_name="gemini-1-0-pro-002-zero-shot",
    params=params,
    metrics=metrics_zero_shot
)


## Few Shot Run

In [15]:
system_prompt_few_shot = f"""TASK:
Classify the text into ONLY one of the following classes [business, entertainment, politics, sport, tech].

CLASSES:
- business
- entertainment
- politics
- sport
- tech

INSTRUCTIONS:
- Respond with ONLY one class.
- You MUST use the exact word from the list above.
- DO NOT create or use any other classes.
- CAREFULLY analyze the text before choosing the best-fitting category from [business, entertainment, politics, sport, tech].

EXAMPLES:
- EXAMPLE 1:
    <user>
    {train.loc[train["label_text"] == "business", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "business", "label_text"].iloc[10]}

- EXAMPLE 2:
    <user>
    {train.loc[train["label_text"] == "entertainment", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "entertainment", "label_text"].iloc[10]}

- EXAMPLE 3:
    <user>
    {train.loc[train["label_text"] == "politics", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "politics", "label_text"].iloc[10]}

- EXAMPLE 4:
    <user>
    {train.loc[train["label_text"] == "sport", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "sport", "label_text"].iloc[10]}

- EXAMPLE 5:
    <user>
    {train.loc[train["label_text"] == "tech", "text"].iloc[10]}
    <model>
    {train.loc[train["label_text"] == "tech", "label_text"].iloc[10]}

"""



In [16]:
gem_pro_1_model_few = GenerativeModel(
    "gemini-1.0-pro-002",  # e.g. gemini-1.5-pro-001, gemini-1.5-flash-001
    system_instruction=[system_prompt_few_shot],
    generation_config=generation_config,
    safety_settings=safety_settings,
)



In [17]:
predictions_few_shot = batch_predict(
    messages=messages_to_predict, model=gem_pro_1_model_few, max_workers=4
)


 73%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                                              | 546/750 [01:47<00:32,  6.35it/s]

Backing off 0.26890771365423805 seconds after 1 tries.
Backing off 0.9422217454189964 seconds after 1 tries.
Backing off 0.8150422317077969 seconds after 1 tries.
Backing off 0.4605671658684256 seconds after 2 tries.
Backing off 0.14978914968609525 seconds after 1 tries.
Backing off 1.2035893774552682 seconds after 2 tries.
Backing off 1.7151571237631733 seconds after 2 tries.
Backing off 0.6597188680496289 seconds after 3 tries.
Backing off 1.484487978325736 seconds after 2 tries.
Backing off 5.060036139123027 seconds after 4 tries.
Backing off 2.7616803284188873 seconds after 3 tries.
Backing off 3.5946661469924486 seconds after 5 tries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 750/750 [02:32<00:00,  4.91it/s]


In [19]:

df_evals["gem1.0-few-shot_predictions"] = predictions_few_shot
print(f"Total few shot predictions: {len(predictions_few_shot)}")
metrics_few_shot = evaluate_predictions(
    df_evals.copy(),
    target_column = "label_text",
    prediction_column = "gem1.0-few-shot_predictions",
    postprocessing = True
)
print(metrics_few_shot)


Total few shot predictions: 750
{'accuracy': 0.8973333333333333, 'weighted precision': 0.9073532470702639, 'weighted recall': 0.8973333333333333, 'macro f1 score': 0.7456228821363435, 'micro f1 score': 0.8973333333333333, 'business_f1_score': 0.8730650154798761, 'entertainment_f1_score': 0.927007299270073, 'politics_f1_score': 0.8689655172413793, 'sport_f1_score': 0.9613259668508287, 'tech_f1_score': 0.8433734939759037}


/home/rajan/.jupyter_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rajan/.jupyter_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rajan/.jupyter_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rajan/.jupyter_venv/li

In [20]:
# Log Experiment with zero-shot prompt with Gemini 1.0 pro
params = {
    "model": "gemini-1.0-pro-002",
    "adaptation_type": "in-context few-shot",
    "temperature": 0,
    "max_output_tokens": 10,
}

experiment_manager.log_run(
    run_name="gemini-1-0-pro-002-few-shot",
    params=params,
    metrics=metrics_few_shot
)

Associating projects/28368249660/locations/us-central1/metadataStores/default/contexts/student-experiment-1-gemini-1-0-pro-002-few-shot to Experiment: student-experiment-1


## Fine Tuning (Supervised)

In [50]:
#####################################
# Fine Tuning Helper Functions
#####################################
def create_gemini_messages(
    text: str, label: str, system_prompt: Optional[str] = None
) -> dict:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(
        [
            {"role": "user", "content": text},
            {"role": "model", "content": label},
        ]
    )
    return {"messages": messages}


def prepare_tuning_dataset_from_df(
    tuning_df: pd.DataFrame, system_prompt: Optional[str] = None
) -> pd.DataFrame:
    """
    Prepares a tuning dataset from a pandas DataFrame for Gemini fine-tuning.
    Args:
        tuning_df: A pandas DataFrame with columns "text" and "label_text".
        system_prompt: An optional system prompt for zero-shot learning.
    Returns:
        A pandas DataFrame containing the data in the Gemini tuning format.
    """
    tuning_dataset = [
        create_gemini_messages(row["text"], row["label_text"], system_prompt)
        for _, row in tuning_df.iterrows()
    ]
    return pd.DataFrame(tuning_dataset)


from google.cloud import storage
import json
def validate_gemini_tuning_jsonl(gcs_jsonl_path: str) -> List[Dict]:
    """
    Validates a JSONL file on Google Cloud Storage against the Gemini tuning format.

    Args:
        gcs_jsonl_path: The GCS path to the JSONL file.

    Returns:
        A list of dictionaries representing the errors found in the file.
        Each dictionary has the following structure:
        {
            "error_type": "Error description",
            "row_index": The index of the row where the error occurred,
            "message": The error message
        }
    """

    errors = []
    storage_client = storage.Client()
    blob = storage.Blob.from_string(uri=gcs_jsonl_path, client=storage_client)

    with blob.open("r") as f:
        for row_index, line in enumerate(f):
            try:
                data = json.loads(line)
                # Check for the presence of the "messages" key
                if "messages" not in data:
                    errors.append(
                        {
                            "error_type": "Missing 'messages' key",
                            "row_index": row_index,
                            "message": f"Row {row_index} is missing the 'messages' key.",
                        }
                    )
                    continue

                messages = data["messages"]
                # Check if "messages" is a list
                if not isinstance(messages, list):
                    errors.append(
                        {
                            "error_type": "Invalid 'messages' type",
                            "row_index": row_index,
                            "message": f"Row {row_index}: 'messages' is not a list.",
                        }
                    )
                    continue

                # Validate each message in the "messages" list
                for message_index, message in enumerate(messages):
                    if not isinstance(message, dict):
                        errors.append(
                            {
                                "error_type": "Invalid message format",
                                "row_index": row_index,
                                "message": f"""Row {row_index},
                            message {message_index}: Message is not a dictionary.""",
                            }
                        )
                        continue

                    # Check for required keys in each message dictionary
                    if "role" not in message or "content" not in message:
                        errors.append(
                            {
                                "error_type": "Missing 'role' or 'content' key",
                                "row_index": row_index,
                                "message": f"Row {row_index}, message {message_index}: "
                                "Missing 'role' or 'content' key.",
                            }
                        )
                        continue

                    # Check for valid role values
                    if message["role"] not in ["system", "user", "model"]:
                        errors.append(
                            {
                                "error_type": "Invalid 'role' value",
                                "row_index": row_index,
                                "message": f"""Row {row_index}, message {message_index}:
                            Invalid 'role' value. Expected 'system', 'user', or 'model'.""",
                            }
                        )
                        continue

            except json.JSONDecodeError as e:
                errors.append(
                    {
                        "error_type": "JSON Decode Error",
                        "row_index": row_index,
                        "message": f"Row {row_index}: JSON decoding error: {e}",
                    }
                )

    return errors

In [34]:
tuning_gemini_df = prepare_tuning_dataset_from_df(
    tuning_df=train, system_prompt=system_prompt_zero_shot
)

In [37]:
tuning_gemini_df.head()

,messages
0,"[{'role': 'system', 'content': 'TASK: Classify..."
1,"[{'role': 'system', 'content': 'TASK: Classify..."
2,"[{'role': 'system', 'content': 'TASK: Classify..."
3,"[{'role': 'system', 'content': 'TASK: Classify..."
4,"[{'role': 'system', 'content': 'TASK: Classify..."


In [40]:
BUCKET_NAME = "rtc-ai-2024020-bucket-9" 
BUCKET_URI = f"gs://{BUCKET_NAME}"

In [41]:
!gsutil mb -l $LOCATION -p $PROJECT_ID $BUCKET_URI

Creating gs://rtc-ai-2024020-bucket-9/...


In [43]:
! pip install --quiet gcsfs

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.1.0 requires fsspec[http]<=2024.9.0,>=2023.1.0, but you have fsspec 2024.10.0 which is incompatible.


In [44]:
# store tuning dataset in GCS
tuning_data_gcs_path = f"gs://{BUCKET_NAME}/tuning_experiments/tuning_dataset_gemini.jsonl"

tuning_gemini_df.to_json(tuning_data_gcs_path, orient="records", lines=True)

In [51]:
validate_gemini_tuning_jsonl(gcs_jsonl_path=tuning_data_gcs_path)

[]

In [52]:
validation_gemini_df = prepare_tuning_dataset_from_df(
    tuning_df=val, system_prompt=system_prompt_zero_shot
)
validation_gemini_df.head()

,messages
0,"[{'role': 'system', 'content': 'TASK: Classify..."
1,"[{'role': 'system', 'content': 'TASK: Classify..."
2,"[{'role': 'system', 'content': 'TASK: Classify..."
3,"[{'role': 'system', 'content': 'TASK: Classify..."
4,"[{'role': 'system', 'content': 'TASK: Classify..."


In [54]:
validation_gemini_df.__len__()

249

In [55]:
# store validation dataset in GCS
validation_data_gcs_path = f"gs://{BUCKET_NAME}/tuning_experiments/validation_dataset_gemini.jsonl"  # @param {type: "string"}
validation_gemini_df.to_json(validation_data_gcs_path, orient="records", lines=True)

In [56]:
validate_gemini_tuning_jsonl(gcs_jsonl_path=validation_data_gcs_path)

[]

In [57]:
tuned_model_name = "studentmodel1"
epochs = 4  # @param
learning_rate_multiplier = 1  # @param
adapter_size = 1  # @param

In [58]:
sft_tuning_job = sft.train(
    tuned_model_display_name=tuned_model_name,
    source_model="gemini-1.0-pro-002",
    train_dataset=tuning_data_gcs_path,
    # Optional:
    validation_dataset=validation_data_gcs_path,
    epochs=epochs,
    learning_rate_multiplier=learning_rate_multiplier,
    adapter_size=adapter_size,
)

Creating SupervisedTuningJob
SupervisedTuningJob created. Resource name: projects/28368249660/locations/us-central1/tuningJobs/1318630379497717760
To use this SupervisedTuningJob in another session:
tuning_job = sft.SupervisedTuningJob('projects/28368249660/locations/us-central1/tuningJobs/1318630379497717760')
View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/1318630379497717760?project=28368249660


In [59]:
# Get the tuning job info.
sft_tuning_job.to_dict()

{'name': 'projects/28368249660/locations/us-central1/tuningJobs/1318630379497717760',
 'tunedModelDisplayName': 'studentmodel1',
 'baseModel': 'gemini-1.0-pro-002',
 'supervisedTuningSpec': {'trainingDatasetUri': 'gs://rtc-ai-2024020-bucket-9/tuning_experiments/tuning_dataset_gemini.jsonl',
  'validationDatasetUri': 'gs://rtc-ai-2024020-bucket-9/tuning_experiments/validation_dataset_gemini.jsonl',
  'hyperParameters': {'epochCount': '4',
   'learningRateMultiplier': 1.0,
   'adapterSize': 'ADAPTER_SIZE_ONE'}},
 'state': 'JOB_STATE_PENDING',
 'createTime': '2024-11-09T15:38:17.965810Z',
 'updateTime': '2024-11-09T15:38:17.965810Z'}

In [60]:
# Get the resource name of the tuning job
sft_tuning_job_name = sft_tuning_job.resource_name
sft_tuning_job_name

'projects/28368249660/locations/us-central1/tuningJobs/1318630379497717760'

## Fine Tuned Model Testing

In [73]:
from vertexai.preview.tuning import sft

# Get tuning job
TUNING_JOB_ID = "1318630379497717760"  # @param example 952462564720115710
sft_tuning_job = sft.SupervisedTuningJob(
    f"projects/{PROJECT_ID}/locations/{LOCATION}/tuningJobs/{TUNING_JOB_ID}"
)

In [75]:
# tuned model endpoint name
tuned_model_endpoint_name = sft_tuning_job.tuned_model_endpoint_name
tuned_model_endpoint_name

'projects/28368249660/locations/us-central1/endpoints/5257021278606327808'

In [76]:
tuned_gemini_pro = GenerativeModel(
    tuned_model_endpoint_name,
    system_instruction=[system_prompt_zero_shot],
    generation_config=generation_config,
    safety_settings=safety_settings,
)

In [77]:
response = tuned_gemini_pro.generate_content([test["text"].iloc[4]], stream=False)

In [78]:
print("predicted", response.text)
print("ground truth", test["label_text"].iloc[4])

predicted politics
ground truth politics


In [79]:
# Get the list of messages to predict
messages_to_predict = test["text"].to_list()
# Compute the predictions using the zero-shot prompt
predictions_tuned_model = batch_predict(
    messages=messages_to_predict, model=tuned_gemini_pro, max_workers=4
)

 50%|█████████████████████████████████████████████████████████████████████████████████████▌                                                                                      | 373/750 [01:14<01:00,  6.19it/s]

Backing off 0.09745858039385347 seconds after 1 tries.
Backing off 0.5405268317491546 seconds after 1 tries.
Backing off 1.833426261104805 seconds after 2 tries.
Backing off 0.0747830947233109 seconds after 1 tries.
Backing off 0.8197670722286717 seconds after 1 tries.
Backing off 1.4643782358681023 seconds after 2 tries.
Backing off 1.0206031244933305 seconds after 2 tries.
Backing off 1.2676835891860871 seconds after 2 tries.
Backing off 1.7690975806863611 seconds after 3 tries.
Backing off 1.0083339117620316 seconds after 3 tries.
Backing off 0.4385870401846299 seconds after 3 tries.
Backing off 0.14461250565373263 seconds after 3 tries.
Backing off 2.342143507587352 seconds after 4 tries.
Backing off 0.2935128390991153 seconds after 4 tries.
Backing off 5.183553374341321 seconds after 4 tries.
Backing off 2.1160674629901646 seconds after 5 tries.
Backing off 6.202787427570273 seconds after 4 tries.
Backing off 6.398643698576935 seconds after 5 tries.
Backing off 22.085080176846045 

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 750/750 [02:40<00:00,  4.67it/s]


In [80]:
df_evals["tuned-gem1.0-ep4-lrm1-rank4"] = predictions_tuned_model


In [81]:
metrics_tuned_gemini = evaluate_predictions(
    df_evals.copy(),
    target_column="label_text",
    prediction_column="tuned-gem1.0-ep4-lrm1-rank4",
    postprocessing=True,
)
metrics_tuned_gemini

{'accuracy': 0.9746666666666667,
 'weighted precision': 0.9747225509533202,
 'weighted recall': 0.9746666666666667,
 'macro f1 score': 0.9734753811428682,
 'micro f1 score': 0.9746666666666667,
 'business_f1_score': 0.9673590504451038,
 'entertainment_f1_score': 0.9694656488549618,
 'politics_f1_score': 0.9655172413793104,
 'sport_f1_score': 1.0,
 'tech_f1_score': 0.965034965034965}

In [82]:
# Log Experiment with zero-shot prompt with Gemini 1.0 pro
params = {
    "model": "studentmodel1",
    "adaptation_type": "in-context zero-shot",
    "temperature": 0,
    "max_output_tokens": 10,
}

experiment_manager.log_run(
    run_name="studentmodel1-zero-shot",
    params=params,
    metrics=metrics_tuned_gemini
)

Associating projects/28368249660/locations/us-central1/metadataStores/default/contexts/student-experiment-1-studentmodel1-zero-shot to Experiment: student-experiment-1


## Experiment Metrics Analysis

In [87]:
df_experiments = experiment_manager.get_experiments_data_frame()

In [88]:
df_experiments

,experiment_name,run_name,run_type,state,param.model,param.adaptation_type,param.temperature,param.max_output_tokens,metric.tech_f1_score,metric.business_f1_score,metric.weighted recall,metric.weighted precision,metric.politics_f1_score,metric.accuracy,metric.sport_f1_score,metric.macro f1 score,metric.micro f1 score,metric.entertainment_f1_score
0,student-experiment-1,studentmodel1-zero-shot,system.ExperimentRun,COMPLETE,studentmodel1,in-context zero-shot,0.0,10.0,0.965035,0.967359,0.974667,0.974723,0.965517,0.974667,1.000000,0.973475,0.974667,0.969466
1,student-experiment-1,mytunedmodel2-596637168630235136-zero-shot,system.ExperimentRun,COMPLETE,mytunedmodel2-596637168630235136,in-context zero-shot,0.0,10.0,0.978723,0.973294,0.982667,0.982708,0.973384,0.982667,1.000000,0.982050,0.982667,0.984848
2,student-experiment-1,gemini-1-0-pro-002-few-shot,system.ExperimentRun,COMPLETE,gemini-1.0-pro-002,in-context few-shot,0.0,10.0,0.843373,0.873065,0.897333,0.907353,0.868966,0.897333,0.961326,0.745623,0.897333,0.927007
3,student-experiment-1,gemini-1-0-pro-002-zero-shot,system.ExperimentRun,COMPLETE,gemini-1.0-pro-002,in-context zero-shot,0.0,10.0,0.889734,0.896552,0.914667,0.920829,0.875862,0.914667,0.966851,0.912266,0.914667,0.932331
